# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content page, for one client, on
one day. Grain: report_date × client_hash_id × content_hash_id, from
fact_content_daily_performance.

**Time window:** a mid-panel month, month=2026-03. I avoided the
_sample file on purpose — I checked it and confirmed it's exactly June
2026 (2026-06-01 to 2026-06-30, 11,694,072 rows), which is the sealed
final month reserved as a test window, not for developing label logic.

Verified below with three queries.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Create a DuckDB secret for Hugging Face authentication
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

print("DuckDB connected, HF secret created.")

Working dir: /content/flyrank-ml-internship
DuckDB connected, HF secret created.


In [4]:
test_query = """
SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
"""
result = con.execute(test_query).df()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0   11694072 2026-06-01 2026-06-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** gsc_avg_position, impressions, clicks, ctr, sessions,
word_count, freshness/age signals — all knowable before any decision
point.

**Label/proxy:** a trend_direction-derived proxy, or a stronger
future-window label (prior 90 days → decline in next 30 days) built
from report_date windows.

**Context:** client_hash_id, content_hash_id, url_hash_id,
keyword_hash_id — used only for joining, grouping, and grouped
train/test splits. Never used as model features.

**Excluded (with why):**
- trend_direction and trend_pct — trend_direction is computed from
  trend_pct, so if my label is proxy-derived from trend_direction,
  using trend_pct as a feature would leak the answer directly.
- Any FlyRank product decision flags (health_score, priority_score,
  action_type) — not shipped in this data; even if rebuilt, I wouldn't
  use them as features since it would just teach a model to copy an
  existing rule.
- GA4-derived features (sessions, engagement_rate, scroll_rate) for
  rows where ga4_data_available is not TRUE — confirmed via query that
  only 4.2% of March rows (413,966 of 9,841,378) have GA4 data
  available. Using GA4 fields without checking this flag would
  silently treat "not tracked yet" as "zero engagement."

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
query1 = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
result1 = con.execute(query1).df()
print("Grain check (should be empty):")
print(result1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, cnt]
Index: []


In [8]:
query2 = """
SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
"""
result2 = con.execute(query2).df()
print(result2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [9]:
query3 = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
"""
result3 = con.execute(query3).df()
print(result3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me the following:

- **GA4 coverage is sparse and uneven.** Only 4.2% of March 2026 rows
  have ga4_data_available = TRUE. Any engagement-based feature
  (sessions, scroll_rate) is only meaningful for a small, non-random
  slice of pages — I can't assume the rest have "zero engagement,"
  only that they weren't tracked yet.

- **Unbalanced client history.** Different clients started tracking at
  different times (checked via dim_clients.gsc_data_start /
  ga4_data_start in the docs). A global calendar window like "March
  2026" doesn't mean every client has equal history — some may be
  newly onboarded mid-month.

- **Window overlaps for query-level data.** fact_content_query_90d is
  a fixed 90-day window that can overlap a target window I define —
  I have to align windows carefully before joining, per the skill
  guide's warning, or I'd risk leaking future information into
  features.

- **This is observational, not causal.** Even with a clean data
  contract, nothing here can prove that refreshing a page caused a
  recovery — only that certain signals are associated with decline or
  growth.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.